In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

In [2]:
df_train = pd.read_csv("Datasets/UNSW_NB15_training-set.csv")
df_test  = pd.read_csv("Datasets/UNSW_NB15_testing-set.csv")

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)

df_train.head()

Train shape: (175341, 45)
Test shape : (82332, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


In [3]:
drop_cols = ['id', 'attack_cat']

df_train = df_train.drop(columns=drop_cols)
df_test  = df_test.drop(columns=drop_cols)

In [4]:
X_train = df_train.drop(columns=['label'])
y_train = df_train['label']

X_test = df_test.drop(columns=['label'])
y_test = df_test['label']

print(X_train.shape, X_test.shape)

(175341, 42) (82332, 42)


In [5]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_features = X_train.select_dtypes(include=['int64','float64']).columns
cat_features = X_train.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

In [6]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=False,   # faster
    random_state=42
)

In [7]:
from sklearn.pipeline import Pipeline

svm_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', svm_model)
])

In [8]:
start_time = time.time()

svm_pipeline.fit(X_train, y_train)

training_time = time.time() - start_time
print("Training Time (s):", round(training_time, 2))

Training Time (s): 1929.24


In [9]:
y_pred = svm_pipeline.predict(X_test)
y_scores = svm_pipeline.decision_function(X_test)

In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_scores))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy : 0.8643297867171938
Precision: 0.8318503263910476
Recall   : 0.9445204270713845
F1 Score : 0.8846122061071856
ROC-AUC  : 0.9597147835429183

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.77      0.84     37000
           1       0.83      0.94      0.88     45332

    accuracy                           0.86     82332
   macro avg       0.88      0.86      0.86     82332
weighted avg       0.87      0.86      0.86     82332

